# Feature Extraction in Text — Full Pipeline

This notebook runs the complete, hand-built pipeline end to end on the
documents in `demo.csv`:

1. Load raw text from CSV
2. Preprocess (lowercase, strip punctuation/numbers, tokenize, remove
   stopwords, stem)
3. POS tagging
4. Morphological analysis
5. Lemmatization (vs. stemming)
6. Vocabulary + Bag of Words / Binary BoW
7. One-hot encoding (tokens and POS tags)
8. TF-IDF
9. N-grams

All logic comes from `preprocessing.py`, `linguistic_features.py`, and
`features.py` in this project — no scikit-learn, nltk, or spaCy.


## 1. Setup

In [1]:
import pandas as pd

from preprocessing import (
    preprocess,
    lowercase,
    remove_punctuation_and_numbers,
    tokenize,
    remove_stopwords,
    stem_word,
    stem_tokens,
)
from linguistic_features import (
    pos_tag,
    pos_tag_word,
    morphological_analysis,
    lemmatize_word,
    lemmatize_tokens,
)
from features import (
    build_vocabulary,
    bag_of_words,
    binary_bow,
    tf_idf,
    n_grams,
    one_hot_encode_tokens,
    one_hot_encode_categories,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)


## 2. Load raw text from CSV

Reads `demo.csv`, which has a single `text` column with one document per
row.


In [2]:
data = pd.read_csv("demo.csv")
corpus = data["text"].tolist()

print(f"Loaded {len(corpus)} documents from demo.csv\n")
for i, doc in enumerate(corpus):
    print(f"Doc {i}: {doc}")


Loaded 8 documents from demo.csv

Doc 0: The cat sat on the mat and looked at the dog.
Doc 1: Dogs are running quickly in the park every morning.
Doc 2: The quick brown fox jumps over the lazy dog.
Doc 3: Cats and dogs are popular pets around the world.
Doc 4: She quickly finished reading the interesting book.
Doc 5: The park was full of happy dogs and playful cats.
Doc 6: A curious fox wandered through the quiet forest at night.
Doc 7: Books about dogs and cats are popular with young readers.


## 3. Preprocessing

`preprocess()` runs lowercase -> strip punctuation/numbers -> tokenize ->
remove stopwords -> stem, in one call. We also keep a lighter "raw tokens"
version per document (lowercased and tokenized, but *not* stopword-stripped
or stemmed) for the linguistic analysis steps below, since POS tagging and
morphology need function words and full word forms to work with.


In [3]:
processed_docs = [preprocess(doc) for doc in corpus]
raw_tokens_per_doc = [
    tokenize(remove_punctuation_and_numbers(lowercase(doc))) for doc in corpus
]

print("=== Tokens after full preprocessing (stopwords removed, stemmed) ===")
for i, tokens in enumerate(processed_docs):
    print(f"Doc {i}: {tokens}")


=== Tokens after full preprocessing (stopwords removed, stemmed) ===
Doc 0: ['cat', 'sat', 'mat', 'look', 'dog']
Doc 1: ['dog', 'runn', 'quick', 'park', 'every', 'morn']
Doc 2: ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']
Doc 3: ['cat', 'dog', 'popular', 'pet', 'around', 'world']
Doc 4: ['quick', 'finish', 'read', 'interest', 'book']
Doc 5: ['park', 'full', 'happy', 'dog', 'playful', 'cat']
Doc 6: ['curiou', 'fox', 'wander', 'quiet', 'forest', 'night']
Doc 7: ['book', 'dog', 'cat', 'popular', 'young', 'reader']


## 4. POS Tagging

Rule-based tagging (closed-class lexicon + suffix rules) on the raw tokens
of every document, shown as one combined table.


In [4]:
pos_rows = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    for word, tag in pos_tag(tokens):
        pos_rows.append({"doc": doc_index, "word": word, "pos_tag": tag})

pos_df = pd.DataFrame(pos_rows)
pos_df


,doc,word,pos_tag
0,0,the,DET
1,0,cat,NOUN
2,0,sat,NOUN
3,0,on,PREP
4,0,the,DET
...,...,...,...
70,7,are,AUX
71,7,popular,NOUN
72,7,with,PREP
73,7,young,NOUN


## 5. Morphological Analysis

Per-word shape and inflection features (length, vowel/consonant counts,
prefix/suffix, plural/gerund/past-tense/comparative/superlative flags) for
every document, combined into one table.


In [5]:
morph_frames = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    doc_morph = morphological_analysis(tokens)
    doc_morph.insert(0, "doc", doc_index)
    morph_frames.append(doc_morph)

morph_df = pd.concat(morph_frames, ignore_index=True)
morph_df


,doc,word,length,num_vowels,num_consonants,prefix3,suffix3,is_capitalized,is_plural,is_gerund,is_past_tense,is_comparative,is_superlative
0,0,the,3,1,2,the,the,False,False,False,False,False,False
1,0,cat,3,1,2,cat,cat,False,False,False,False,False,False
2,0,sat,3,1,2,sat,sat,False,False,False,False,False,False
3,0,on,2,1,1,on,on,False,False,False,False,False,False
4,0,the,3,1,2,the,the,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,7,are,3,2,1,are,are,False,False,False,False,False,False
71,7,popular,7,3,4,pop,lar,False,False,False,False,False,False
72,7,with,4,1,3,wit,ith,False,False,False,False,False,False
73,7,young,5,2,3,you,ung,False,False,False,False,False,False


## 6. Lemmatization vs. Stemming

Both reduce a word to a base form, but the stemmer just chops suffixes
(sometimes producing fragments that aren't real words), while the
lemmatizer aims to return an actual dictionary word. Comparison shown on
stopword-free tokens from every document.


In [6]:
lemma_rows = []
for doc_index, tokens in enumerate(raw_tokens_per_doc):
    filtered = remove_stopwords(tokens)
    for word in filtered:
        lemma_rows.append({
            "doc": doc_index,
            "word": word,
            "stem": stem_word(word),
            "lemma": lemmatize_word(word),
        })

lemma_df = pd.DataFrame(lemma_rows)
lemma_df


,doc,word,stem,lemma
0,0,cat,cat,cat
1,0,sat,sat,sat
2,0,mat,mat,mat
3,0,looked,look,look
4,0,dog,dog,dog
5,1,dogs,dog,dog
6,1,running,runn,run
7,1,quickly,quick,quick
8,1,park,park,park
9,1,every,every,every


## 7. Vocabulary

Built from the fully preprocessed (stopword-free, stemmed) tokens.


In [7]:
vocab = build_vocabulary(processed_docs)
print(f"Vocabulary size: {len(vocab)}")
vocab


Vocabulary size: 32


['around',
 'book',
 'brown',
 'cat',
 'curiou',
 'dog',
 'every',
 'finish',
 'forest',
 'fox',
 'full',
 'happy',
 'interest',
 'jump',
 'lazy',
 'look',
 'mat',
 'morn',
 'night',
 'park',
 'pet',
 'playful',
 'popular',
 'quick',
 'quiet',
 'read',
 'reader',
 'runn',
 'sat',
 'wander',
 'world',
 'young']

## 8. Bag of Words

In [8]:
bow_df = bag_of_words(processed_docs, vocab)
bow_df


,around,book,brown,cat,curiou,dog,every,finish,forest,fox,full,happy,interest,jump,lazy,...,morn,night,park,pet,playful,popular,quick,quiet,read,reader,runn,sat,wander,world,young
0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,1,0,0,0,1,0,0,0,0
2,0,0,1,0,0,1,0,0,0,1,0,0,0,1,1,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
5,0,0,0,1,0,1,0,0,0,0,1,1,0,0,0,...,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0
7,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1


## 9. Binary Bag of Words

In [9]:
binary_df = binary_bow(processed_docs, vocab)
binary_df


,around,book,brown,cat,curiou,dog,every,finish,forest,fox,full,happy,interest,jump,lazy,...,morn,night,park,pet,playful,popular,quick,quiet,read,reader,runn,sat,wander,world,young
0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,1,0,0,0,1,0,0,0,0
2,0,0,1,0,0,1,0,0,0,1,0,0,0,1,1,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
5,0,0,0,1,0,1,0,0,0,0,1,1,0,0,0,...,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0
7,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1


## 10. One-Hot Encoding

Two flavors: one-hot per token *position* in a document (preserves order,
unlike bag-of-words), and a generic one-hot encoding of category labels
(here, the POS tags of Doc 0).


In [10]:
print("=== One-hot encoding of tokens (Doc 0) ===")
one_hot_tokens_df = one_hot_encode_tokens(processed_docs[0], vocab)
one_hot_tokens_df


=== One-hot encoding of tokens (Doc 0) ===


,around,book,brown,cat,curiou,dog,every,finish,forest,fox,full,happy,interest,jump,lazy,...,morn,night,park,pet,playful,popular,quick,quiet,read,reader,runn,sat,wander,world,young
0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
print("=== One-hot encoding of POS tags (Doc 0) ===")
pos_tags_doc0 = [tag for _, tag in pos_tag(raw_tokens_per_doc[0])]
one_hot_pos_df = one_hot_encode_categories(pos_tags_doc0)
one_hot_pos_df


=== One-hot encoding of POS tags (Doc 0) ===


,CONJ,DET,NOUN,PREP,VERB
0,0,1,0,0,0
1,0,0,1,0,0
2,0,0,1,0,0
3,0,0,0,1,0
4,0,1,0,0,0
5,0,0,1,0,0
6,1,0,0,0,0
7,0,0,0,0,1
8,0,0,0,1,0
9,0,1,0,0,0


## 11. TF-IDF

In [12]:
tfidf_df = tf_idf(processed_docs, vocab)
tfidf_df.round(3)


,around,book,brown,cat,curiou,dog,every,finish,forest,fox,full,happy,interest,jump,lazy,...,morn,night,park,pet,playful,popular,quick,quiet,read,reader,runn,sat,wander,world,young
0,0.000,0.000,0.000,0.693,0.000,0.288,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.079,0.000,0.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.288,2.079,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,2.079,0.000,1.386,0.000,0.000,0.000,0.981,0.000,0.000,0.000,2.079,0.000,0.000,0.000,0.000
2,0.000,0.000,2.079,0.000,0.000,0.288,0.000,0.000,0.000,1.386,0.000,0.000,0.000,2.079,2.079,...,0.000,0.000,0.000,0.000,0.000,0.000,0.981,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,2.079,0.000,0.000,0.693,0.000,0.288,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,2.079,0.000,1.386,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.079,0.000
4,0.000,1.386,0.000,0.000,0.000,0.000,0.000,2.079,0.000,0.000,0.000,0.000,2.079,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.981,0.000,2.079,0.000,0.000,0.000,0.000,0.000,0.000
5,0.000,0.000,0.000,0.693,0.000,0.288,0.000,0.000,0.000,0.000,2.079,2.079,0.000,0.000,0.000,...,0.000,0.000,1.386,0.000,2.079,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6,0.000,0.000,0.000,0.000,2.079,0.000,0.000,0.000,2.079,1.386,0.000,0.000,0.000,0.000,0.000,...,0.000,2.079,0.000,0.000,0.000,0.000,0.000,2.079,0.000,0.000,0.000,0.000,2.079,0.000,0.000
7,0.000,1.386,0.000,0.693,0.000,0.288,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,1.386,0.000,0.000,0.000,2.079,0.000,0.000,0.000,0.000,2.079


## 12. N-grams

Bigrams and trigrams for every document, built from the fully preprocessed
tokens.


In [13]:
for i, tokens in enumerate(processed_docs):
    print(f"Doc {i}")
    print("  Bigrams: ", n_grams(tokens, 2))
    print("  Trigrams:", n_grams(tokens, 3))


Doc 0
  Bigrams:  [('cat', 'sat'), ('sat', 'mat'), ('mat', 'look'), ('look', 'dog')]
  Trigrams: [('cat', 'sat', 'mat'), ('sat', 'mat', 'look'), ('mat', 'look', 'dog')]
Doc 1
  Bigrams:  [('dog', 'runn'), ('runn', 'quick'), ('quick', 'park'), ('park', 'every'), ('every', 'morn')]
  Trigrams: [('dog', 'runn', 'quick'), ('runn', 'quick', 'park'), ('quick', 'park', 'every'), ('park', 'every', 'morn')]
Doc 2
  Bigrams:  [('quick', 'brown'), ('brown', 'fox'), ('fox', 'jump'), ('jump', 'lazy'), ('lazy', 'dog')]
  Trigrams: [('quick', 'brown', 'fox'), ('brown', 'fox', 'jump'), ('fox', 'jump', 'lazy'), ('jump', 'lazy', 'dog')]
Doc 3
  Bigrams:  [('cat', 'dog'), ('dog', 'popular'), ('popular', 'pet'), ('pet', 'around'), ('around', 'world')]
  Trigrams: [('cat', 'dog', 'popular'), ('dog', 'popular', 'pet'), ('popular', 'pet', 'around'), ('pet', 'around', 'world')]
Doc 4
  Bigrams:  [('quick', 'finish'), ('finish', 'read'), ('read', 'interest'), ('interest', 'book')]
  Trigrams: [('quick', 'finis

## Summary

```
raw text -> lowercase -> remove punctuation/numbers -> tokenize
         -> remove stopwords -> stem/lemmatize
         -> [POS tagging / morphological analysis on the side]
         -> feature extraction (BoW / one-hot / TF-IDF / n-grams)
```

Every step above is implemented from scratch in `preprocessing.py`,
`linguistic_features.py`, and `features.py` using only `numpy` and
`pandas`. See `report.md` for the theory behind each step.
